<font color='tomato'><font color="#CC3D3D"><p>
# Baseline Code (v2.0)

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import pickle
import gzip
import gc
import re
import warnings; warnings.filterwarnings("ignore")
import seaborn as sns
import matplotlib.pylab as plt
from matplotlib import font_manager, rc
from tqdm import tqdm, tqdm_notebook
%matplotlib inline

from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.cluster import KMeans


from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

import shap # v2.0부터 추가

In [ ]:
import gzip
import pickle

# 압축 해제해서 데이터 불러오기
with gzip.open('12210346_Original_.zip', 'rb') as f:
    X_train, y_train, X_test, ID_test, cat_features, num_features = pickle.load(f)

In [ ]:
# 파일 이름 설정
tuned_models = './oof_hardvoting_tuned_models12210357.csv' 
MinMaxScaler = './oof_hardvoting_MinMaxScaler12210414.csv'
Log = './oof_hardvoting_tuned_models_Log12210437.csv'  

# 결과 파일들을 데이터프레임으로 불러오기
tuned_models = pd.read_csv(tuned_models)
MinMaxScaler = pd.read_csv(MinMaxScaler) 
Log = pd.read_csv(Log)

# 예측값을 가져오기 (가정: 마지막 열이 예측값)
final_predictions_tuned_models = tuned_models.iloc[:, -1].values
final_predictions_MinMaxScaler = MinMaxScaler.iloc[:, -1].values
final_predictions_Log = Log.iloc[:, -1].values

In [ ]:
# 예측값들의 평균 계산
avg_predictions = (final_predictions_tuned_models + final_predictions_MinMaxScaler + final_predictions_Log) / 3

# 평균값을 기준으로 0 또는 1로 예측
threshold = 0.5  # 임계값 설정
final_predictions = np.where(avg_predictions >= threshold, 1, 0)

print(final_predictions)

In [ ]:
t = pd.Timestamp.now()
fname = f"each_file__oof_hardvoting_{t.month:02}{t.day:02}{t.hour:02}{t.minute:02}.csv"
pd.DataFrame({'ID': ID_test, 'STATUS': final_predictions}).to_csv(fname, index=False)
print(f"'{fname}' is ready to submit.")